In [2]:
## 1. Setup & Imports
import os
import numpy as np
import xarray as xr
import torch
import torch.nn as nn
import torch.nn.functional as F
# import lightning as L
from torch.utils.data import Dataset, DataLoader
# import terratorch  # fine-tuning toolkit for geospatial foundation models


In [ ]:
# notebook: radar_to_terra_embedding.ipynb


print("Torch version:", torch.__version__)

## 2. Load your radar Zarr dataset
zarr_path = "KVBX_preserve_500m.zarr"
ds = xr.open_zarr(zarr_path, consolidated=False)
da = ds["reflectivity"]  # dims: (time, z, y, x)
print(da)

## 3. Prepare radar stacks as input patches
# Example: stack N time slices per sample, flatten vertical dimension
T = 6   # number of time scans per sample
C = T     # for simplicity 1 channel per scan (or use multiple fields)
Z = da.z.size
H = da.y.size
W = da.x.size

# Create a simple dataset for tokenizer
class RadarStackDataset(Dataset):
    def __init__(self, da, times, stack_size=T):
        self.da = da
        self.times = times  # list of indices
        self.stack_size = stack_size

    def __len__(self):
        return len(self.times) - self.stack_size

    def __getitem__(self, idx):
        t0 = self.times[idx]
        # select stack_size scans
        arr = self.da.isel(time=slice(t0, t0+self.stack_size)).values  # shape (T, Z, H, W)
        arr = arr.astype(np.float32)
        # condense vertical or choose one altitude slice if needed
        # e.g., select first Z level: arr = arr[:, 0, :, :]
        # reshape to (C, H, W)
        arr2d = arr.reshape(self.stack_size, H, W)
        # simple label placeholder (you’ll use gauge data in real training)
        label = 0.0
        return torch.from_numpy(arr2d), torch.tensor(label, dtype=torch.float32)

times = list(range(da.time.size))
radar_ds = RadarStackDataset(da, times, stack_size=T)
radar_loader = DataLoader(radar_ds, batch_size=8, shuffle=True, num_workers=4)

## 4. Define your CNN tokenizer
class RadarTokenizerCNN(nn.Module):
    def __init__(self, in_channels=6, latent_dim=1024):  # e.g., 6 time-frames or channels
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, 32, kernel_size=3, padding=0)  # input (6, H, W)
        self.bn1   = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(kernel_size=2)
        self.drop1 = nn.Dropout2d(p=0.25)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=0)
        self.bn2   = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(kernel_size=2)
        self.drop2 = nn.Dropout2d(p=0.25)

        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=0)
        self.bn3   = nn.BatchNorm2d(128)
        self.pool3 = nn.MaxPool2d(kernel_size=2)
        self.drop3 = nn.Dropout2d(p=0.25)

        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, padding=0)
        self.bn4   = nn.BatchNorm2d(256)
        self.pool4 = nn.MaxPool2d(kernel_size=2)
        self.drop4 = nn.Dropout2d(p=0.25)

        # compute feature map size (you’ll need to calculate given H, W)
        # then flatten
        final_H = H // 16
        final_W = W // 16
        self.fc1 = nn.Linear(256 * final_H * final_W, 256)
        self.projection = nn.Linear(256, latent_dim)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.pool1(x)
        x = self.drop1(x)

        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool2(x)
        x = self.drop2(x)

        x = F.relu(self.bn3(self.conv3(x)))
        x = self.pool3(x)
        x = self.drop3(x)

        x = F.relu(self.bn4(self.conv4(x)))
        x = self.pool4(x)
        x = self.drop4(x)

        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        embed = self.projection(x)
        return embed

## 5. Train tokenizer (supervised) — simple Lightning module
class TokenizerModule(pl.LightningModule):
    def __init__(self, model, lr=1e-4):
        super().__init__()
        self.model = model
        self.lr = lr
        self.loss_fn = nn.MSELoss()

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        embed = self.model(x)
        # for supervised pre-training: predict rainfall (dummy here)
        pred = embed.mean(dim=1)  # simple projection to scalar
        loss = self.loss_fn(pred, y)
        self.log("train_loss", loss)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.model.parameters(), lr=self.lr)

tokenizer_module = TokenizerModule(tokenizer_model)
trainer = pl.Trainer(max_epochs=10, gpus=1)
trainer.fit(tokenizer_module, radar_loader)

## 6. Extract embeddings for each sample
embeddings = []
coords = []
for x, y in radar_loader:
    embed = tokenizer_model(x)
    embeddings.append(embed.detach().cpu().numpy())
    # store associated coords/time for alignment
embeddings = np.concatenate(embeddings, axis=0)
# save as .npy or integrate into xarray for geospatial alignment
np.save("radar_embeddings.npy", embeddings)

## 7. Integrate with TerraMind via TerraTorch
from terratorch import BACKBONE_REGISTRY, TaskFactory

# build TerraMind backbone
terra_backbone = BACKBONE_REGISTRY.build("terratorch_terramind_v1", pretrained=True)

# Suppose task: pixel-wise regression of rainfall map
task = TaskFactory.build("segmentation_regression", backbone=terra_backbone, num_classes=1)

# Extend data module to include radar embeddings as additional channel
# You’ll need to create a TerrTorch DataModule that reads your geospatial raster stack (sat + radar embed) and target rainfall maps
from terratorch.data import GeoDataModule

class MyGeoDataModule(GeoDataModule):
    def __init__(self, ...):
        super().__init__(...)
        # handle additional radar embed channel

    def prepare_data(self):
        # load satellite, radar embedding, static layers, target rainfall map

    def setup(self, stage=None):
        # create train/val/test splits

dm = MyGeoDataModule(...)
trainer2 = pl.Trainer(max_epochs=20, gpus=1)
trainer2.fit(task, dm)

## 8. Inference & embedding visualization
# After fine-tuning, run inference
pred = task.predict(...)
# Visualize overlay on map

